In [5]:
# ===== 代码块 1：纯文本方式合并两个 CSV（不清理、不解析）=====
import os

base_dir = r".\数据\上市公司子公司联营合营情况表"
file_a = os.path.join(base_dir, "STK_NotesSubJoint.csv")
file_b = os.path.join(base_dir, "STK_NotesSubJoint1.csv")
merged_csv = os.path.join(base_dir, "STK_NotesSubJoint_merged.csv")

# 纯拼接：保留 A 的表头；B 跳过表头；其余原样写入
with open(file_a, "r", encoding="utf-8", errors="replace", newline="") as fa, \
     open(file_b, "r", encoding="utf-8", errors="replace", newline="") as fb, \
     open(merged_csv, "w", encoding="utf-8", errors="replace", newline="") as fo:

    header_a = fa.readline()
    if not header_a:
        raise RuntimeError(f"{file_a} is empty")
    fo.write(header_a)

    for line in fa:
        fo.write(line)

    header_b = fb.readline()  # skip B header
    if not header_b:
        raise RuntimeError(f"{file_b} is empty")

    if header_b.strip() != header_a.strip():
        print("[WARN] A/B header not identical (still merged by skipping B header).")

    for line in fb:
        fo.write(line)

print("[OK] merged ->", merged_csv)


[OK] merged -> .\数据\上市公司子公司联营合营情况表\STK_NotesSubJoint_merged.csv


In [3]:
# ===== 代码块 2（替换版）：生成更“像手动粘贴”的 xlsx，兼容爱企查 =====
import os
import csv
import math
import re
from openpyxl import Workbook
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE

merged_csv = r".\数据\上市公司子公司联营合营情况表\STK_NotesSubJoint_merged.csv"
out_dir = r"./数据/aiqicha_query_files"
os.makedirs(out_dir, exist_ok=True)

batch_size = 10000
target_col = "RalatedParty"

# 额外清理：常见“看不见但会让上传失败”的字符（零宽字符、BOM、不可见分隔符等）
ZERO_WIDTH_RE = re.compile(r"[\u200b\u200c\u200d\u2060\ufeff]")

def sanitize_for_upload(s: str) -> str:
    # 1) 去掉 openpyxl 不允许的控制字符
    s = ILLEGAL_CHARACTERS_RE.sub("", s)
    # 2) 去掉零宽字符/BOM
    s = ZERO_WIDTH_RE.sub("", s)
    # 不做 strip（你要求不筛选/不处理名字），但如果存在纯空串就返回空
    return s

# 1) 找列索引
with open(merged_csv, "r", encoding="utf-8", errors="replace", newline="") as f:
    reader = csv.reader(f)
    header = next(reader)
if target_col not in header:
    raise KeyError(f"找不到列 {target_col}，表头列为：{header}")
idx = header.index(target_col)
print(f"[OK] column '{target_col}' index = {idx}")

# 2) 提取公司名并去重（行坏跳过）
uniq = set()
bad_rows = 0
total_rows = 0

with open(merged_csv, "r", encoding="utf-8", errors="replace", newline="") as f:
    reader = csv.reader(f)
    _ = next(reader)  # skip header
    for row in reader:
        total_rows += 1
        try:
            if len(row) <= idx:
                bad_rows += 1
                continue
            name = row[idx]
            if name is None or name == "":
                continue
            name = sanitize_for_upload(str(name))
            if name == "":
                continue
            uniq.add(name)
        except Exception:
            bad_rows += 1
            continue

names = sorted(uniq)
print(f"[STAT] total_rows_scanned={total_rows:,}, bad_rows_skipped={bad_rows:,}, unique_names={len(names):,}")

# 3) 用 openpyxl 生成“极简 xlsx”（最像手动粘贴的文件结构）
def write_xlsx_minimal(path: str, values: list[str]):
    wb = Workbook()
    ws = wb.active
    ws.title = "Sheet1"
    # 表头严格是 企业名称
    ws.cell(row=1, column=1, value="企业名称")
    # 逐行写入（纯文本）
    for i, v in enumerate(values, start=2):
        ws.cell(row=i, column=1, value=v)
    wb.save(path)

n_batches = math.ceil(len(names) / batch_size)
for i in range(n_batches):
    part = names[i*batch_size:(i+1)*batch_size]
    out_path = os.path.join(out_dir, f"aiqicha_batch_{i+1:02d}.xlsx")
    write_xlsx_minimal(out_path, part)
    print(f"[OUT] {out_path} rows={len(part):,}")


[OK] column 'RalatedParty' index = 2
[STAT] total_rows_scanned=1,658,230, bad_rows_skipped=0, unique_names=333,995
[OUT] ./数据/aiqicha_query_files\aiqicha_batch_01.xlsx rows=10,000
[OUT] ./数据/aiqicha_query_files\aiqicha_batch_02.xlsx rows=10,000
[OUT] ./数据/aiqicha_query_files\aiqicha_batch_03.xlsx rows=10,000
[OUT] ./数据/aiqicha_query_files\aiqicha_batch_04.xlsx rows=10,000
[OUT] ./数据/aiqicha_query_files\aiqicha_batch_05.xlsx rows=10,000
[OUT] ./数据/aiqicha_query_files\aiqicha_batch_06.xlsx rows=10,000
[OUT] ./数据/aiqicha_query_files\aiqicha_batch_07.xlsx rows=10,000
[OUT] ./数据/aiqicha_query_files\aiqicha_batch_08.xlsx rows=10,000
[OUT] ./数据/aiqicha_query_files\aiqicha_batch_09.xlsx rows=10,000
[OUT] ./数据/aiqicha_query_files\aiqicha_batch_10.xlsx rows=10,000
[OUT] ./数据/aiqicha_query_files\aiqicha_batch_11.xlsx rows=10,000
[OUT] ./数据/aiqicha_query_files\aiqicha_batch_12.xlsx rows=10,000
[OUT] ./数据/aiqicha_query_files\aiqicha_batch_13.xlsx rows=10,000
[OUT] ./数据/aiqicha_query_files\aiqicha_b

In [6]:
#把爱企查返回的xls转换成xlsx格式，方便python处理

import os
import glob

# 如果没装过：
# !pip install pywin32

import win32com.client as win32

dir_path = r".\数据\爱企查结果"
xls_files = sorted(glob.glob(os.path.join(dir_path, "*.xls")))

print("xls files:", len(xls_files))
if not xls_files:
    raise FileNotFoundError("目录下没有 .xls 文件")

excel = win32.Dispatch("Excel.Application")
excel.Visible = False
excel.DisplayAlerts = False

converted = 0
failed = []

try:
    for xls in xls_files:
        xlsx = os.path.splitext(xls)[0] + ".xlsx"
        # 如果已经存在 xlsx，就跳过
        if os.path.exists(xlsx):
            continue
        try:
            wb = excel.Workbooks.Open(os.path.abspath(xls))
            # 51 = xlOpenXMLWorkbook (.xlsx)
            wb.SaveAs(os.path.abspath(xlsx), FileFormat=51)
            wb.Close(False)
            converted += 1
        except Exception as e:
            failed.append((xls, str(e)))
finally:
    excel.Quit()

print(f"[OK] converted = {converted}")
if failed:
    print(f"[WARN] failed = {len(failed)} (show first 5)")
    for p, e in failed[:5]:
        print("  ", p, "=>", e)


xls files: 34
[OK] converted = 34


In [7]:
import os
import glob
import pandas as pd

dir_path = r".\数据\爱企查结果"
paths = sorted(glob.glob(os.path.join(dir_path, "*.xlsx")))

print("xlsx files:", len(paths))
if not paths:
    raise FileNotFoundError("目录下没有 .xlsx 文件（请先运行上一步转换）")

def pick_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"找不到列，候选={candidates}，实际={df.columns.tolist()}")

dfs = []
bad = []

for p in paths:
    try:
        df = pd.read_excel(p, header=2, engine="openpyxl")  # 第3行标题
        col_match = pick_col(df, ["匹配结果", "匹配状态"])
        df[col_match] = df[col_match].astype(str).str.strip()
        df = df[df[col_match] == "成功"].copy()
        df["_source_file"] = os.path.basename(p)
        dfs.append(df)
    except Exception as e:
        bad.append((p, str(e)))

print(f"[STAT] ok={len(dfs)}, bad={len(bad)}")
if bad:
    print("[WARN] show first 5 bad files:")
    for p, e in bad[:5]:
        print("  ", p, "=>", e)

if not dfs:
    raise RuntimeError("所有文件都读取失败，请把任意一个 xlsx 发我看一下表头行。")

merged = pd.concat(dfs, ignore_index=True)

# 1) 保存合并全量
merged_out = os.path.join(dir_path, "爱企查结果_merged.xlsx")
merged.to_excel(merged_out, index=False)
print("[OK] saved:", merged_out, "rows=", len(merged))

# 2) 只保留 企业名称 + 统一社会信用代码
col_name = pick_col(merged, ["企业名称", "公司名称", "名称"])
col_ucc  = pick_col(merged, ["统一社会信用代码", "统一社会信用码", "社会信用代码", "信用代码"])

mapping = merged[[col_name, col_ucc]].copy()
mapping.columns = ["企业名称", "统一社会信用代码"]
mapping = mapping.dropna(subset=["企业名称", "统一社会信用代码"]).drop_duplicates()

map_out = os.path.join(dir_path, "上市公司子公司对应统一社会信用代码.xlsx")
mapping.to_excel(map_out, index=False)
print("[OK] saved:", map_out, "rows=", len(mapping))


xlsx files: 34
[STAT] ok=34, bad=0
[OK] saved: .\数据\爱企查结果\爱企查结果_merged.xlsx rows= 262467
[OK] saved: .\数据\爱企查结果\上市公司子公司对应统一社会信用代码.xlsx rows= 242786


In [8]:
#转成读取更快的csv格式，方便后续处理
import os
import pandas as pd

# ===== 配置 =====
base_dir = r".\数据\爱企查结果"

xlsx_files = [
    os.path.join(base_dir, "爱企查结果_merged.xlsx"),
    os.path.join(base_dir, "上市公司子公司对应统一社会信用代码.xlsx"),
]

# ===== 转换 =====
for xlsx_path in xlsx_files:
    if not os.path.exists(xlsx_path):
        raise FileNotFoundError(f"找不到文件: {xlsx_path}")

    print(f"[READ] {xlsx_path}")
    df = pd.read_excel(xlsx_path, engine="openpyxl")

    csv_path = os.path.splitext(xlsx_path)[0] + ".csv"

    # UTF-8 with BOM，保证中文 & 平台兼容
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")

    print(f"[OK]  -> {csv_path}  rows={len(df):,}, cols={df.shape[1]}")


[READ] .\数据\爱企查结果\爱企查结果_merged.xlsx
[OK]  -> .\数据\爱企查结果\爱企查结果_merged.csv  rows=262,467, cols=29
[READ] .\数据\爱企查结果\上市公司子公司对应统一社会信用代码.xlsx
[OK]  -> .\数据\爱企查结果\上市公司子公司对应统一社会信用代码.csv  rows=242,786, cols=2


In [16]:
# ============================================================
# 本脚本用于从上市公司年度基础信息表（STK_LISTEDCOINFOANL.csv）
# 中构建“上市公司统一社会信用代码映射表”，并完成必要的数据清洗与整理。
#
# 主要处理步骤说明：
# 1. 读取 CSV 原始数据（UTF-8 编码），并对结构性脏行直接丢弃
#    （如列数不一致、分隔符错误等），保证整体流程不中断。
#
# 2. 数据清洗：
#    - 解析 EndDate 为日期格式，无法解析的记录视为脏数据并剔除；
#    - 剔除统一社会信用代码为空的记录；
#    - 对股票代码（Symbol）、股票简称（ShortName）等字段做基础去空处理。
#
# 3. 基于 EndDate 计算年份（Year），并在股票代码（Symbol）层面：
#    - 统计该股票出现的最早年份（FirstYear）与最晚年份（LastYear）；
#    - 判断该股票历史上是否曾出现过 ST 类名称（EverST）。
#
# 4. 当同一股票代码在历史上对应多个股票简称时：
#    - 优先选取时间上最新（EndDate 最大）的记录；
#    - 若最新名称包含 ST，则优先选取“最新但不包含 ST”的名称；
#    - 若全部名称均为 ST，则退而求其次，选取最新记录。
#
# 5. 最终生成以股票代码（stkid）为唯一键的汇总表，
#    输出字段包括：
#      - stkid              : 股票代码
#      - shortname          : 选定的股票简称（非 ST 优先、时间靠后）
#      - SocialCreditCode   : 统一社会信用代码
#      - EverST             : 是否历史上出现过 ST（0/1）
#      - FirstYear          : 最早出现年份
#      - LastYear           : 最晚出现年份
#
# 6. 结果以 UTF-8-SIG 编码导出为 CSV 文件：
#    “上市公司统一社会信用代码.csv”，
#    供后续与子公司、专利、财务等数据进行稳定 join 使用。
# ============================================================

import os
import pandas as pd

# ===== 路径 =====
base_dir = r".\数据\上市公司基本信息年度表"
in_csv = os.path.join(base_dir, "STK_LISTEDCOINFOANL.csv")
out_csv = os.path.join(base_dir, "上市公司统一社会信用代码.csv")

# ===== 1) 读取：utf-8 + 丢弃结构性坏行 =====
df = pd.read_csv(
    in_csv,
    dtype=str,
    encoding="utf-8",
    low_memory=False,
    on_bad_lines="skip"  # 你确认脏行直接丢即可
)
df.columns = [c.strip() for c in df.columns]

need_cols = ["Symbol", "ShortName", "EndDate", "ListedCoID", "SocialCreditCode"]
missing = [c for c in need_cols if c not in df.columns]
if missing:
    raise KeyError(f"缺少列: {missing}，实际列: {df.columns.tolist()}")

# ===== 2) 清洗：日期合法 + 信用代码非空 =====
df["Symbol"] = df["Symbol"].astype(str).str.strip()
df["ShortName"] = df["ShortName"].astype(str).str.strip()
df["SocialCreditCode"] = df["SocialCreditCode"].astype(str).str.strip()

# EndDate 解析（允许 2000/12/31 或 2000-12-31 等）
df["EndDate_dt"] = pd.to_datetime(df["EndDate"], errors="coerce")
df = df[df["EndDate_dt"].notna()].copy()

# 信用代码为空的丢弃
df = df[df["SocialCreditCode"].notna() & (df["SocialCreditCode"] != "")].copy()
df = df[df["Symbol"].notna() & (df["Symbol"] != "")].copy()

# 计算年份
df["Year"] = df["EndDate_dt"].dt.year.astype(int)

# ===== 3) ST 识别 & EverST =====
df["IsSTName"] = df["ShortName"].str.upper().str.contains("ST", na=False)
ever_st = df.groupby("Symbol")["IsSTName"].max().astype(int).rename("EverST")

# ===== 4) 每个 stkid 的最早/最晚年份 =====
year_range = df.groupby("Symbol")["Year"].agg(FirstYear="min", LastYear="max")

# ===== 5) 为每个 Symbol 选 “最终简称 + 信用代码”（偏向后期，且尽量不含ST）=====
def pick_latest_nonst_else_latest(g: pd.DataFrame):
    g = g.sort_values("EndDate_dt")  # 时间升序
    g_nonst = g[~g["IsSTName"]]
    row = g_nonst.iloc[-1] if len(g_nonst) > 0 else g.iloc[-1]
    return pd.Series({
        "ShortName": row["ShortName"],
        "SocialCreditCode": row["SocialCreditCode"],
        "ListedCoID": row["ListedCoID"],
    })

picked = df.groupby("Symbol", as_index=True).apply(pick_latest_nonst_else_latest)

# 合并 EverST + 年份范围
result = picked.join(ever_st, how="left").join(year_range, how="left").reset_index()

# ===== 6) 输出 6 列（你要的 4 列 + 最早/最晚年份）=====
result = result.rename(columns={
    "Symbol": "stkid",
    "ShortName": "shortname",
    "SocialCreditCode": "SocialCreditCode"
})

result = result[["stkid", "shortname", "SocialCreditCode", "EverST", "FirstYear", "LastYear"]]
result = result.sort_values(["stkid"]).reset_index(drop=True)

result.to_csv(out_csv, index=False, encoding="utf-8-sig")
print(f"[OK] saved -> {out_csv}")
print(result.head(10))
print(f"[STAT] rows={len(result):,}, unique_stkid={result['stkid'].nunique():,}")


[OK] saved -> .\数据\上市公司基本信息年度表\上市公司统一社会信用代码.csv
    stkid shortname    SocialCreditCode  EverST  FirstYear  LastYear
0  000001      平安银行  91440300192185379H       0       2000      2024
1  000002       万科A  91440300192181490G       0       2000      2024
2  000003    PT 金田A                 nan       1       2000      2001
3  000004      国华网安  91440300192441969E       1       2000      2024
4  000005      世纪星源  914403006188470942       1       2000      2022
5  000006      深振业A  91440300618831041G       0       2000      2024
6  000007       全新好  9144030019217870XW       1       2000      2024
7  000008      神州高铁  91110000192184333K       1       2000      2024
8  000009      中国宝安  9144030019219665XD       0       2000      2024
9  000010      美丽生态  91110000192181597U       1       2000      2024
[STAT] rows=5,732, unique_stkid=5,732


C:\Users\LuHaoqi\AppData\Local\Temp\ipykernel_8332\1803846604.py:58: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  picked = df.groupby("Symbol", as_index=True).apply(pick_latest_nonst_else_latest)


In [19]:
#检查“子公司名称 → 统一社会信用代码”这张映射表里，是否存在「同一个名称对应多个不同信用代码」的情况。
# 结果：存在，可能是一个公司先注销了再有，也有可能是完全不相关的两个公司

import os
import pandas as pd

# ===== 路径 =====
base_dir = r".\数据\爱企查结果"
map_csv = os.path.join(base_dir, "上市公司子公司对应统一社会信用代码.csv")

# ===== 1) 读取映射表 =====
df = pd.read_csv(
    map_csv,
    dtype=str,
    encoding="utf-8",
    low_memory=False,
    on_bad_lines="skip"
)

# 标准化列名
df.columns = [c.strip() for c in df.columns]

need_cols = ["企业名称", "统一社会信用代码"]
missing = [c for c in need_cols if c not in df.columns]
if missing:
    raise KeyError(f"缺少列 {missing}，实际列为：{df.columns.tolist()}")

# 基础清洗
df["企业名称"] = df["企业名称"].astype(str).str.strip()
df["统一社会信用代码"] = df["统一社会信用代码"].astype(str).str.strip()

df = df[
    (df["企业名称"] != "") &
    (df["统一社会信用代码"] != "")
].copy()

print(f"[STAT] total rows after basic clean = {len(df):,}")

# ===== 2) 检查：同名是否对应多个不同信用代码 =====
# 对每个 企业名称，统计唯一信用代码数量
name_ucc_cnt = (
    df.groupby("企业名称")["统一社会信用代码"]
      .nunique()
      .rename("ucc_cnt")
)

# 找出 ucc_cnt > 1 的名称
dup_name = name_ucc_cnt[name_ucc_cnt > 1]

print(f"[STAT] 企业名称总数 = {name_ucc_cnt.shape[0]:,}")
print(f"[STAT] 同名对应多个信用代码的企业数 = {dup_name.shape[0]:,}")

# ===== 3) 输出这些“问题名称”的明细，方便人工检查 =====
if len(dup_name) > 0:
    dup_detail = (
        df[df["企业名称"].isin(dup_name.index)]
        .sort_values(["企业名称", "统一社会信用代码"])
        .reset_index(drop=True)
    )

    out_path = os.path.join(base_dir, "子公司同名多统一社会信用代码明细.csv")
    dup_detail.to_csv(out_path, index=False, encoding="utf-8-sig")

    print(f"[OK] 同名多码明细已输出 -> {out_path}")
    print(dup_detail)
else:
    print("[OK] 未发现“同名对应多个统一社会信用代码”的情况")


[STAT] total rows after basic clean = 242,786
[STAT] 企业名称总数 = 242,770
[STAT] 同名对应多个信用代码的企业数 = 16
[OK] 同名多码明细已输出 -> .\数据\爱企查结果\子公司同名多统一社会信用代码明细.csv
                企业名称            统一社会信用代码
0      东莞市康正轧辊设备有限公司  91440300398520718J
1      东莞市康正轧辊设备有限公司  914419000507044525
2      东莞新时达智能装备有限公司  914403007852825374
3      东莞新时达智能装备有限公司  91441900MA522L7Y9K
4       北京七彩柠檬科技有限公司  91110106306339111B
5       北京七彩柠檬科技有限公司  91110112MAEX0R1J8A
6       双鸥电器（陕西）有限公司  91610302MA6XH8BC0P
7       双鸥电器（陕西）有限公司  916103047304159109
8         吉林金泰投资有限公司  91220101691020288M
9         吉林金泰投资有限公司  91220221MABMLNEJ96
10        天津铁海物流有限公司  91120102MA078MDD69
11        天津铁海物流有限公司  91120116103082184J
12      宁波东力传动科技有限公司  91330201563876938R
13      宁波东力传动科技有限公司  91330201MA7CLFMKX6
14  广东中科英华材料科技发展有限公司  91440101MA9WE2W40K
15  广东中科英华材料科技发展有限公司  91440300732948761H
16      广东美的楼宇科技有限公司  91440606714820860N
17      广东美的楼宇科技有限公司  91440606MA58RYUL66
18            杭州明州医院  52330109MJ8842244F
19            杭州明州医院  91330109MAB

In [22]:
# ============================================================
# 全流程：生成「上市公司（包括所有子公司）各年度的统一社会信用代码列表.csv」
# 输出 4 列：证券ID-公司简称-年份-统一社会信用代码列表
#
# 输入：
# 1) 数据\上市公司基本信息年度表\上市公司统一社会信用代码.csv
#    - 含 stkid / shortname / SocialCreditCode / FirstYear / LastYear
# 2) 数据\上市公司子公司对应统一社会信用代码.csv
#    - 含 企业名称 / 统一社会信用代码（可能一名多码）
# 3) 数据\上市公司子公司联营合营情况表\STK_NotesSubJoint_merged.csv
#    - 只用 Symbol / EndDate / RalatedParty / Relationship（Relationship 全保留）
#
# 性能策略：
# - 子公司名称 → 统一社会信用代码字符串：一次性 groupby 拼接成 dict（O(1) 查询）
# - 子公司明细表：只读必要列 + chunksize 分块读取 + 每块内 groupby 拼接后累加到 dict
# - 本任务不拆分多码，不做 code 级去重；仅做字符串拼接与基本分号压缩
# ============================================================

import os
import re
import pandas as pd

# ===== 路径配置（按你当前项目结构）=====
base_data_dir = r".\数据"

parent_csv = os.path.join(base_data_dir, r"上市公司基本信息年度表\上市公司统一社会信用代码.csv")
subs_map_csv = os.path.join(
    base_data_dir,
    "爱企查结果",
    "上市公司子公司对应统一社会信用代码.csv"
)
subjoint_csv = os.path.join(base_data_dir, r"上市公司子公司联营合营情况表\STK_NotesSubJoint_merged.csv")

out_csv = os.path.join(base_data_dir, "上市公司（包括所有子公司）各年度的统一社会信用代码列表.csv")

SEP = ";"                 # 拼接分隔符（不拆分）
CHUNKSIZE = 300_000       # 子公司明细分块大小（可根据内存调大/调小）

# ============================================================
# 0) 小工具：压缩多余分号
# ============================================================
def normalize_seps(s: str) -> str:
    if s is None:
        return ""
    s = str(s)
    # 去掉首尾分号 + 连续分号压缩
    s = s.strip(SEP)
    while SEP + SEP in s:
        s = s.replace(SEP + SEP, SEP)
    return s

# ============================================================
# 1) 读取母公司表：stkid / shortname / SocialCreditCode / FirstYear / LastYear
# ============================================================
parent = pd.read_csv(
    parent_csv,
    dtype=str,
    encoding="utf-8",
    low_memory=False,
    on_bad_lines="skip",
)
parent.columns = [c.strip() for c in parent.columns]

need_parent_cols = ["stkid", "shortname", "SocialCreditCode", "FirstYear", "LastYear"]
missing = [c for c in need_parent_cols if c not in parent.columns]
if missing:
    raise KeyError(f"母公司表缺少列 {missing}，实际列：{parent.columns.tolist()}")

parent["stkid"] = parent["stkid"].astype(str).str.strip()
parent["shortname"] = parent["shortname"].astype(str).str.strip()
parent["SocialCreditCode"] = parent["SocialCreditCode"].astype(str).str.strip()

parent["FirstYear"] = pd.to_numeric(parent["FirstYear"], errors="coerce")
parent["LastYear"] = pd.to_numeric(parent["LastYear"], errors="coerce")

parent = parent[
    parent["stkid"].notna() & (parent["stkid"] != "") &
    parent["SocialCreditCode"].notna() & (parent["SocialCreditCode"] != "") &
    parent["FirstYear"].notna() & parent["LastYear"].notna()
].copy()

parent["FirstYear"] = parent["FirstYear"].astype(int)
parent["LastYear"] = parent["LastYear"].astype(int)

print(f"[OK] loaded parent rows={len(parent):,}, unique_stkid={parent['stkid'].nunique():,}")

# ============================================================
# 2) 读取子公司映射表：企业名称 -> 统一社会信用代码字符串（同名多码直接拼接）
#    注意：本任务不拆分，不做 code 级去重；仅在“同名多码”聚合时去重避免重复写入
# ============================================================
subs_map = pd.read_csv(
    subs_map_csv,
    dtype=str,
    encoding="utf-8",
    low_memory=False,
    on_bad_lines="skip",
)
subs_map.columns = [c.strip() for c in subs_map.columns]
need_map_cols = ["企业名称", "统一社会信用代码"]
missing = [c for c in need_map_cols if c not in subs_map.columns]
if missing:
    raise KeyError(f"子公司映射表缺少列 {missing}，实际列：{subs_map.columns.tolist()}")

subs_map["企业名称"] = subs_map["企业名称"].astype(str).str.strip()
subs_map["统一社会信用代码"] = subs_map["统一社会信用代码"].astype(str).str.strip()
subs_map = subs_map[(subs_map["企业名称"] != "") & (subs_map["统一社会信用代码"] != "")].copy()

# 同名多码：聚合为一个字符串（这里为了避免完全重复，先 set 去重再拼接）
name_to_ucc_str = (
    subs_map.groupby("企业名称")["统一社会信用代码"]
            .apply(lambda s: SEP.join(sorted(set(s))))
            .to_dict()
)

print(f"[OK] built name_to_ucc_str size={len(name_to_ucc_str):,}")

# ============================================================
# 3) 扫描 STK_NotesSubJoint_merged.csv：累积 (Symbol, Year) -> 子公司信用代码字符串拼接
#    - Relationship 全部保留
#    - 只取必要列，分块处理
#    - 子公司名找不到映射：跳过
# ============================================================
usecols = ["Symbol", "EndDate", "RalatedParty", "Relationship"]

child_acc = {}  # key=(Symbol, Year) -> "codeA;codeB;codeC..."（本任务不拆分）

# EndDate 快速提取年份：允许 "YYYY-..." / "YYYY/..." / "YYYYMMDD"；只要前四位是数字即可
year4_re = re.compile(r"^\d{4}$")

chunk_iter = pd.read_csv(
    subjoint_csv,
    usecols=usecols,
    dtype=str,
    encoding="utf-8",
    low_memory=False,
    on_bad_lines="skip",
    chunksize=CHUNKSIZE
)

total_rows = 0
kept_rows = 0
miss_map_rows = 0

for ci, chunk in enumerate(chunk_iter, start=1):
    total_rows += len(chunk)

    # 基础清洗
    chunk["Symbol"] = chunk["Symbol"].astype(str).str.strip()
    chunk["EndDate"] = chunk["EndDate"].astype(str).str.strip()
    chunk["RalatedParty"] = chunk["RalatedParty"].astype(str).str.strip()

    # 提取 Year（只要前四位是数字）
    chunk["Year"] = chunk["EndDate"].str.slice(0, 4)
    chunk = chunk[chunk["Year"].apply(lambda x: bool(year4_re.match(str(x))))].copy()
    chunk["Year"] = chunk["Year"].astype(int)

    # 映射：子公司名称 -> 代码字符串
    chunk["ChildUCCStr"] = chunk["RalatedParty"].map(name_to_ucc_str)

    # 找不到映射的跳过
    miss_map_rows += chunk["ChildUCCStr"].isna().sum()
    chunk = chunk[chunk["ChildUCCStr"].notna()].copy()
    kept_rows += len(chunk)

    if chunk.empty:
        if ci % 10 == 0:
            print(f"[SCAN] chunk {ci}, total_rows={total_rows:,}, kept_rows={kept_rows:,}, acc_keys={len(child_acc):,}")
        continue

    # 每块内按 (Symbol, Year) 聚合成一个拼接字符串，再累加到全局 dict
    grp = (
        chunk.groupby(["Symbol", "Year"])["ChildUCCStr"]
             .apply(lambda s: SEP.join(s.astype(str).tolist()))
    )

    for (sym, yr), s in grp.items():
        key = (sym, int(yr))
        if key in child_acc:
            child_acc[key] = child_acc[key] + SEP + s
        else:
            child_acc[key] = s

    if ci % 10 == 0:
        print(f"[SCAN] chunk {ci}, total_rows={total_rows:,}, kept_rows={kept_rows:,}, acc_keys={len(child_acc):,}")

print(f"[OK] scanned subjoint total_rows={total_rows:,}, kept_rows={kept_rows:,}, miss_map_rows={miss_map_rows:,}, acc_keys={len(child_acc):,}")

# ============================================================
# 4) 生成最终面板：对每个 stkid 的 FirstYear..LastYear 输出一行
#    统一社会信用代码列表 = 母公司代码 + (该年子公司代码串, 若有)
# ============================================================
out_rows = []
for _, r in parent.iterrows():
    stkid = r["stkid"]
    shortname = r["shortname"]
    parent_ucc = normalize_seps(r["SocialCreditCode"])
    y0, y1 = int(r["FirstYear"]), int(r["LastYear"])

    # 防御：年份范围异常则跳过
    if y0 > y1 or y0 < 1900 or y1 > 2100:
        continue

    for y in range(y0, y1 + 1):
        child_str = child_acc.get((stkid, y), "")
        # 本任务：不拆分，只拼接
        ucc_list = parent_ucc
        if child_str:
            ucc_list = ucc_list + SEP + child_str
        ucc_list = normalize_seps(ucc_list)

        out_rows.append([stkid, shortname, y, ucc_list])

out_df = pd.DataFrame(out_rows, columns=["证券ID", "公司简称", "年份", "统一社会信用代码列表"])

out_df.to_csv(out_csv, index=False, encoding="utf-8-sig")
print(f"[OK] saved -> {out_csv}")
print(out_df.head(10))
print(f"[STAT] output_rows={len(out_df):,}, unique_stkid={out_df['证券ID'].nunique():,}, years_range=({out_df['年份'].min()}..{out_df['年份'].max()})")


[OK] loaded parent rows=5,732, unique_stkid=5,732
[OK] built name_to_ucc_str size=242,770
[OK] scanned subjoint total_rows=1,658,230, kept_rows=1,158,588, miss_map_rows=499,642, acc_keys=66,345
[OK] saved -> .\数据\上市公司（包括所有子公司）各年度的统一社会信用代码列表.csv
     证券ID  公司简称    年份                             统一社会信用代码列表
0  000001  平安银行  2000  91440300192185379H;91440300192201076Y
1  000001  平安银行  2001  91440300192185379H;91440300192201076Y
2  000001  平安银行  2002  91440300192185379H;91440300192201076Y
3  000001  平安银行  2003  91440300192185379H;91440300192201076Y
4  000001  平安银行  2004  91440300192185379H;91440300192201076Y
5  000001  平安银行  2005  91440300192185379H;91440300192201076Y
6  000001  平安银行  2006  91440300192185379H;91440300192201076Y
7  000001  平安银行  2007                     91440300192185379H
8  000001  平安银行  2008  91440300192185379H;91370100757467681P
9  000001  平安银行  2009  91440300192185379H;91370100757467681P
[STAT] output_rows=68,857, unique_stkid=5,732, years_range=(2000..2025)


In [23]:
# Check 1｜结构完整性（最基础，但必须做）
import pandas as pd

df = pd.read_csv(
    r".\数据\上市公司（包括所有子公司）各年度的统一社会信用代码列表.csv",
    dtype=str,
    encoding="utf-8"
)

print(df.columns.tolist())
print(df.isna().sum())
print(df.head())


['证券ID', '公司简称', '年份', '统一社会信用代码列表']
证券ID           0
公司简称           0
年份             0
统一社会信用代码列表    85
dtype: int64
     证券ID  公司简称    年份                             统一社会信用代码列表
0  000001  平安银行  2000  91440300192185379H;91440300192201076Y
1  000001  平安银行  2001  91440300192185379H;91440300192201076Y
2  000001  平安银行  2002  91440300192185379H;91440300192201076Y
3  000001  平安银行  2003  91440300192185379H;91440300192201076Y
4  000001  平安银行  2004  91440300192185379H;91440300192201076Y


In [24]:
# Check 2｜年份覆盖是否“合理”（不该断裂）

df["年份"] = df["年份"].astype(int)

# 每个公司：最小年、最大年、行数
year_span = (
    df.groupby("证券ID")["年份"]
      .agg(min_year="min", max_year="max", n_years="count")
)

# 理论上 n_years == max_year - min_year + 1
year_span["expected_years"] = year_span["max_year"] - year_span["min_year"] + 1
year_span["gap"] = year_span["expected_years"] - year_span["n_years"]

year_span["gap"].value_counts().head()


gap
0    5732
Name: count, dtype: int64

In [27]:
# Check 3｜统一社会信用代码数量分布（极端值检查）
def count_codes_safe(s):
    """
    统计统一社会信用代码列表中 code 的数量
    - 非字符串（NaN / float） -> 0
    - 字符串 'nan' / 空串 -> 0
    - 其余按 ; 分割计数
    """
    if not isinstance(s, str):
        return 0

    s = s.strip()
    if not s or s.lower() == "nan":
        return 0

    return sum(1 for x in s.split(";") if x.strip() and x.strip().lower() != "nan")
df["code_cnt"] = df["统一社会信用代码列表"].apply(count_codes_safe)

df["code_cnt"].describe(percentiles=[0.5, 0.9, 0.95, 0.99])


count    68857.000000
mean        17.772456
std         37.570624
min          0.000000
50%          9.000000
90%         37.000000
95%         58.000000
99%        145.000000
max       1387.000000
Name: code_cnt, dtype: float64

In [30]:
# Check 4｜“母公司代码是否始终在列表里”（逻辑一致性）

# 4.1 构造母公司代码映射
parent = pd.read_csv(
    r".\数据\上市公司基本信息年度表\上市公司统一社会信用代码.csv",
    dtype=str,
    encoding="utf-8"
)

parent_ucc = dict(
    zip(parent["stkid"], parent["SocialCreditCode"])
)

# 4.2 检查是否有年份“母公司代码不在列表里”
def parent_missing(row):
    stkid = row["证券ID"]
    parent_code = parent_ucc.get(stkid, "")
    
    # 检查“统一社会信用代码列表”是否为字符串
    ucc_list = row["统一社会信用代码列表"]
    if not isinstance(ucc_list, str):
        return False  # 如果不是字符串，就认为没有匹配项

    return parent_code not in ucc_list.split(";")

df["parent_missing"] = df.apply(parent_missing, axis=1)

# 查看缺失情况
df["parent_missing"].value_counts()



parent_missing
False    68054
True       803
Name: count, dtype: int64

In [34]:
# Check 5｜随机抽样“人工可解释性 check”（最值钱）
df.sample(5, random_state=41)


,证券ID,公司简称,年份,统一社会信用代码列表,code_cnt,parent_missing
35232,300858,科拓生物,2020,91110116754160123E;91110116771593449W;91330702...,13,False
51033,600692,亚通股份,2015,91310000132221817R;91310230134414751X;91310230...,19,False
27853,300118,东方日升,2016,913302001449739014;91330226563879864P;91310000...,8,False
53742,600808,马钢股份,2013,91340000610400837Y;913405007430658766;91340500...,12,False
52166,600740,山西焦化,2002,91140000113273064E;3101011022211;1426001000723...,4,False
